In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os

# ── Build path to data folder ──
data_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'assets', 'data'))

print(f"Looking for data in: {data_dir}")
print(f"Files found: {os.listdir(data_dir)}")

# ── Load Cleaned CSV ──
df = pd.read_csv(os.path.join(data_dir, "Student Study Hours & Academic Performance Survey (Cleaned).csv"))

# Map encoded values back to readable labels
cgpa_map         = {3.75: 'High (3.50–4.00)', 3.25: 'Mid (3.00–3.49)', 2.75: 'Low (2.00–3.00)'}
study_hours_map  = {0.5: '< 1 hour', 1.5: '1–2 hours', 3.5: '3–4 hours', 5.0: '5+ hours'}
social_media_map = {0.5: '< 1 hour', 1.5: '1–2 hours', 3.5: '3–4 hours', 5.0: '5+ hours'}
exam_hours_map   = {1: '< 3 hours', 3: '3–4 hours', 6: '5–7 hours', 8: '8+ hours'}

df['cgpa_label']         = df['cgpa'].map(cgpa_map)
df['study_hours_label']  = df['study_hours_daily'].map(study_hours_map)
df['social_media_label'] = df['social_media_hours'].map(social_media_map)
df['exam_hours_label']   = df['study_hours_exam'].map(exam_hours_map)

# Colour palettes
CGPA_COLORS = {
    'High (3.50–4.00)': '#4D96FF',
    'Mid (3.00–3.49)':  '#FFD93D',
    'Low (2.00–3.00)':  '#FF6B6B'
}
CGPA_ORDER = ['Low (2.00–3.00)', 'Mid (3.00–3.49)', 'High (3.50–4.00)']

print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"\nCGPA distribution:")
print(df['cgpa_label'].value_counts())
print(f"\nLevel of Studies columns:")
print([c for c in df.columns if 'Level' in c])


Looking for data in: c:\Users\andly\Documents\Y2 S2\DVID\Student Academic Performance Dashboard\assets\data
Files found: ['Student Study Hours & Academic Performance Survey  (Cleaned).csv', 'Student Study Hours & Academic Performance Survey (Completed).csv', 'Student Study Hours & Academic Performance Survey (RAW).csv']
Dataset loaded: 88 rows × 40 columns
Columns: ['study_hours_daily', 'study_hours_exam', 'time_management', 'reading_plan', 'note_taking', 'group_study', 'lecture_skip_hours', 'online_resources', 'cgpa', 'subjects_enrolled', 'coursework_score', 'social_media_hours', 'social_media_distraction', 'exercise_impact', 'Gender_Female', 'Gender_Male', "Current Level of Studies_Bachelor's Degree", 'Current Level of Studies_Diploma', 'Current Level of Studies_Foundation', "Current Level of Studies_Master's Degree", 'Type of Institution_Private University', 'Type of Institution_Public University', 'What is your preferred study method ?_Flashcards', 'What is your preferred study met

In [2]:
# ═══════════════════════════════════════════════════════════════
# VISUALISATION 1: Bar Chart
# Average Daily Study Hours by CGPA Category
# ═══════════════════════════════════════════════════════════════

avg_hours = df.groupby('cgpa_label')['study_hours_daily'].mean().reindex(CGPA_ORDER)
counts    = df['cgpa_label'].value_counts().reindex(CGPA_ORDER)

fig = go.Figure()

for cat, val, n in zip(CGPA_ORDER, avg_hours, counts):
    fig.add_trace(go.Bar(
        x=[cat],
        y=[val],
        marker=dict(color=CGPA_COLORS[cat], line=dict(color='white', width=2)),
        text=[f'<b>{val:.2f} hrs</b><br>n={n}'],
        textposition='outside',
        textfont=dict(size=14, family='Arial Black'),
        hovertemplate=(
            '<b>%{x}</b><br>'
            f'Avg Study Hours: {val:.2f} hrs<br>'
            f'Students: {n}<br>'
            '<extra></extra>'
        ),
        name=cat,
        showlegend=True
    ))

fig.update_layout(
    title=dict(
        text='<b>📚 Average Daily Study Hours by CGPA Category</b>',
        x=0.5, xanchor='center',
        font=dict(size=22, color='#1A237E', family='Arial Black')
    ),
    xaxis=dict(
        title=dict(text='<b>CGPA Category</b>', font=dict(size=16, color='#1A237E')),
        showgrid=False,
        tickfont=dict(size=13, color='#2C3E50')
    ),
    yaxis=dict(
        title=dict(text='<b>Average Daily Study Hours</b>', font=dict(size=16, color='#1A237E')),
        showgrid=True, gridcolor='rgba(200,200,200,0.3)',
        tickfont=dict(size=12, color='#2C3E50'),
        range=[0, (avg_hours.max() if avg_hours.notna().any() else 1) * 1.35]
    ),
    plot_bgcolor='rgba(240,248,255,0.5)',
    paper_bgcolor='white',
    hovermode='x unified',
    hoverlabel=dict(bgcolor='white', font_size=13, font_family='Arial'),
    legend=dict(
        orientation='h', yanchor='bottom', y=1.02,
        xanchor='right', x=1,
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='#1A237E', borderwidth=2
    ),
    height=600,
    bargap=0.3
)

fig.show()

In [3]:
# ═══════════════════════════════════════════════════════════════
# VISUALISATION 2: Pie/Donut Chart
# Study Challenge Category Distribution
# ═══════════════════════════════════════════════════════════════

challenge_cols   = [c for c in df.columns if c.startswith('Study Challenge Category_')]
challenge_counts = df[challenge_cols].sum().sort_values(ascending=False)
challenge_labels = [c.replace('Study Challenge Category_', '') for c in challenge_counts.index]

# Add 'Academic Workload' for students with no challenge flagged
no_challenge = (df[challenge_cols].sum(axis=1) == 0).sum()
if no_challenge > 0:
    challenge_counts['Academic Workload'] = no_challenge
    challenge_counts = challenge_counts.sort_values(ascending=False)
    challenge_labels = [c.replace('Study Challenge Category_', '') for c in challenge_counts.index]

pie_colors = ['#4D96FF', '#FFD93D', '#6BCB77', '#FF6B9D', '#9D84B7', '#FF5722', '#00BCD4', '#E91E63']

fig = go.Figure(data=[go.Pie(
    labels=challenge_labels,
    values=challenge_counts.values,
    hole=0.45,
    marker=dict(
        colors=pie_colors[:len(challenge_labels)],
        line=dict(color='white', width=3)
    ),
    textinfo='percent',
    textfont=dict(size=12, family='Arial'),
    hovertemplate='<b>%{label}</b><br>Students: %{value}<br>Percentage: %{percent}<extra></extra>',
    pull=[0.05 if i == 0 else 0 for i in range(len(challenge_labels))],
    rotation=140
)])

fig.add_annotation(
    text=f'<b>{int(challenge_counts.sum())}</b><br><span style="font-size:14px">Responses</span>',
    x=0.5, y=0.5,
    font=dict(size=20, color='#1A237E', family='Arial Black'),
    showarrow=False
)

fig.update_layout(
    title=dict(
        text='<b>🎯 Study Challenge Category Distribution</b>',
        x=0.5, xanchor='center',
        font=dict(size=22, color='#1A237E', family='Arial Black')
    ),
    font=dict(size=13, family='Arial'),
    showlegend=True,
    legend=dict(
        orientation='v', yanchor='middle', y=0.5,
        xanchor='left', x=1.02,
        font=dict(size=12),
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='#1A237E', borderwidth=1
    ),
    paper_bgcolor='white',
    height=700, width=1000,
    hoverlabel=dict(bgcolor='white', font_size=13, font_family='Arial'),
    margin=dict(t=80, b=20, l=30, r=180)
)

fig.show()

In [4]:
# ═══════════════════════════════════════════════════════════════
# VISUALISATION 3: Scatter Plot
# Social Media Hours vs Coursework Score (coloured by CGPA)
# ═══════════════════════════════════════════════════════════════

np.random.seed(42)
plot_df = df[['social_media_hours', 'coursework_score', 'cgpa_label', 'study_hours_daily']].copy()
plot_df['sm_j'] = plot_df['social_media_hours'] + np.random.normal(0, 0.1, len(plot_df))
plot_df['cw_j'] = plot_df['coursework_score']   + np.random.normal(0, 1.2, len(plot_df))

fig = go.Figure()

for label in CGPA_ORDER:
    sub = plot_df[plot_df['cgpa_label'] == label]
    fig.add_trace(go.Scatter(
        x=sub['sm_j'],
        y=sub['cw_j'],
        mode='markers',
        marker=dict(
            color=CGPA_COLORS[label],
            size=12,
            line=dict(color='white', width=1.5),
            opacity=0.75
        ),
        name=label,
        hovertemplate=(
            f'<b>CGPA: {label}</b><br>'
            'Social Media: %{customdata[0]:.1f} hrs/day<br>'
            'Coursework: %{customdata[1]}%<br>'
            'Study Hours: %{customdata[2]:.1f} hrs/day<br>'
            '<extra></extra>'
        ),
        customdata=sub[['social_media_hours', 'coursework_score', 'study_hours_daily']].values
    ))

# Regression trendline
z = np.polyfit(df['social_media_hours'], df['coursework_score'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['social_media_hours'].min() - 0.3, df['social_media_hours'].max() + 0.3, 100)

fig.add_trace(go.Scatter(
    x=x_line, y=p(x_line),
    mode='lines',
    line=dict(color='rgba(220,50,50,0.45)', width=3, dash='dash'),
    name=f'Trend (slope: {z[0]:.2f})',
    hoverinfo='skip'
))

fig.update_layout(
    title=dict(
        text='<b>📱 Social Media Hours vs Coursework Score</b>',
        x=0.5, xanchor='center',
        font=dict(size=22, color='#1A237E', family='Arial Black')
    ),
    xaxis=dict(
        title=dict(text='<b>Social Media Hours per Day</b>', font=dict(size=16, color='#1A237E')),
        showgrid=True, gridcolor='rgba(200,200,200,0.3)',
        tickfont=dict(size=12, color='#2C3E50'),
        tickvals=[0.5, 1.5, 3.5, 5.0],
        ticktext=['< 1 hr', '1–2 hrs', '3–4 hrs', '5+ hrs']
    ),
    yaxis=dict(
        title=dict(text='<b>Coursework Score (%)</b>', font=dict(size=16, color='#1A237E')),
        showgrid=True, gridcolor='rgba(200,200,200,0.3)',
        tickfont=dict(size=12, color='#2C3E50')
    ),
    plot_bgcolor='rgba(240,248,255,0.5)',
    paper_bgcolor='white',
    hovermode='closest',
    hoverlabel=dict(bgcolor='white', font_size=13, font_family='Arial'),
    legend=dict(
        orientation='h', yanchor='top', y=-0.15,
        xanchor='center', x=0.5,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#1A237E', borderwidth=2,
        font=dict(size=13)
    ),
    height=650,
    margin=dict(t=80, b=100)
)

fig.show()


In [5]:
# ═══════════════════════════════════════════════════════════════
# VISUALISATION 4: Line Chart
# Average CGPA by Social Media Usage Hours
# ═══════════════════════════════════════════════════════════════

sm_labels  = ['< 1 hour', '1–2 hours', '3–4 hours', '5+ hours']
sm_numeric = [0.5, 1.5, 3.5, 5.0]

# Safe aggregation — fillna to avoid NaN crash when data is filtered
avg_cgpa  = df.groupby('social_media_hours')['cgpa'].mean().reindex(sm_numeric).fillna(0)
count_sm  = df.groupby('social_media_hours')['cgpa'].count().reindex(sm_numeric).fillna(0).astype(int)
std_cgpa  = df.groupby('social_media_hours')['cgpa'].std().reindex(sm_numeric).fillna(0)

fig = go.Figure()

# Confidence band (mean ± std)
fig.add_trace(go.Scatter(
    x=sm_labels + sm_labels[::-1],
    y=list(avg_cgpa.values + std_cgpa.values) + list((avg_cgpa.values - std_cgpa.values)[::-1]),
    fill='toself',
    fillcolor='rgba(77,150,255,0.12)',
    line=dict(color='rgba(255,255,255,0)'),
    name='± 1 Std Dev',
    hoverinfo='skip'
))

# Main line with labels
fig.add_trace(go.Scatter(
    x=sm_labels,
    y=avg_cgpa.values,
    mode='lines+markers+text',
    line=dict(color='#1565C0', width=4),
    marker=dict(size=16, color='#1565C0', line=dict(color='white', width=3)),
    text=[f'<b>{v:.2f}</b><br>(n={int(n)})' for v, n in zip(avg_cgpa.values, count_sm.values)],
    textposition='top center',
    textfont=dict(size=12, family='Arial Black', color='#1A237E'),
    name='Average CGPA',
    hovertemplate='<b>%{x}</b><br>Avg CGPA: %{y:.2f}<extra></extra>'
))

# Regression trendline — only if data exists
if avg_cgpa.any():
    z_line = np.polyfit(sm_numeric, avg_cgpa.values, 1)
    p_line = np.poly1d(z_line)
    y_trend = [p_line(x) for x in sm_numeric]
    fig.add_trace(go.Scatter(
        x=sm_labels,
        y=y_trend,
        mode='lines',
        line=dict(color='rgba(220,50,50,0.55)', width=2, dash='dot'),
        name=f'Trend (slope: {z_line[0]:.3f})',
        hoverinfo='skip'
    ))

fig.update_layout(
    title=dict(
        text='<b>📉 Average CGPA by Social Media Usage Hours</b>',
        x=0.5, xanchor='center',
        font=dict(size=22, color='#1A237E', family='Arial Black')
    ),
    xaxis=dict(
        title=dict(text='<b>Social Media Hours per Day</b>', font=dict(size=16, color='#1A237E')),
        showgrid=True, gridcolor='rgba(200,200,200,0.3)',
        tickfont=dict(size=13, color='#2C3E50')
    ),
    yaxis=dict(
        title=dict(text='<b>Average CGPA</b>', font=dict(size=16, color='#1A237E')),
        showgrid=True, gridcolor='rgba(200,200,200,0.3)',
        tickfont=dict(size=12, color='#2C3E50'),
        range=[2.4, 4.1]
    ),
    plot_bgcolor='rgba(240,248,255,0.5)',
    paper_bgcolor='white',
    hovermode='x unified',
    hoverlabel=dict(bgcolor='white', font_size=13, font_family='Arial'),
    legend=dict(
        orientation='h', yanchor='bottom', y=1.02,
        xanchor='right', x=1,
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='#1A237E', borderwidth=2
    ),
    height=600
)

fig.show()


In [6]:
# ═══════════════════════════════════════════════════════════════
# VISUALISATION 5: 3D Scatter Plot
# Study Hours × Social Media Hours × Coursework Score
# ═══════════════════════════════════════════════════════════════

np.random.seed(42)
s3d = df[['study_hours_daily', 'social_media_hours', 'coursework_score',
          'cgpa_label', 'cgpa', 'time_management', 'note_taking']].copy()

# Add jitter for better visibility
s3d['x'] = s3d['study_hours_daily']  + np.random.normal(0, 0.15, len(df))
s3d['y'] = s3d['social_media_hours'] + np.random.normal(0, 0.15, len(df))
s3d['z'] = s3d['coursework_score']   + np.random.normal(0, 1.5,  len(df))

symbols = ['diamond', 'square', 'circle']
fig = go.Figure()

for label, sym in zip(CGPA_ORDER, symbols):
    sub = s3d[s3d['cgpa_label'] == label]
    fig.add_trace(go.Scatter3d(
        x=sub['x'],
        y=sub['y'],
        z=sub['z'],
        mode='markers',
        marker=dict(
            color=CGPA_COLORS[label],
            size=7,
            symbol=sym,
            line=dict(color='white', width=1),
            opacity=0.85
        ),
        name=label,
        hovertemplate=(
            f'<b>{label}</b><br>'
            'Daily Study: %{customdata[0]:.1f} hrs<br>'
            'Social Media: %{customdata[1]:.1f} hrs<br>'
            'Coursework: %{customdata[2]}%<br>'
            'Time Mgmt: %{customdata[3]}<br>'
            'Note Taking: %{customdata[4]}<br>'
            '<extra></extra>'
        ),
        customdata=sub[['study_hours_daily', 'social_media_hours',
                         'coursework_score', 'time_management', 'note_taking']].values
    ))

axis_style = dict(
    backgroundcolor='rgba(230,240,250,0.8)',
    gridcolor='rgba(26,35,78,0.25)', gridwidth=2,
    showline=True, linecolor='#1A237E', linewidth=2,
    mirror=True, showbackground=True
)

fig.update_layout(
    title=dict(
        text='<b>🌐 3D View: Study Hours × Social Media × Coursework</b><br>'
             '<sub>Rotate • Zoom • Click Points • Explore!</sub>',
        x=0.5, xanchor='center',
        font=dict(size=20, color='#1A237E', family='Arial Black')
    ),
    scene=dict(
        xaxis=dict(title='Daily Study Hours',    **axis_style),
        yaxis=dict(title='Social Media Hours',   **axis_style),
        zaxis=dict(title='Coursework Score (%)', **axis_style),
        camera=dict(eye=dict(x=1.8, y=1.8, z=1.5))
    ),
    paper_bgcolor='white',
    legend=dict(
        title=dict(text='<b>CGPA Category</b>', font=dict(size=13)),
        orientation='h', yanchor='bottom', y=0.9,
        xanchor='center', x=0.5,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#1A237E', borderwidth=2,
        font=dict(size=13)
    ),
    height=750, width=900,
    hoverlabel=dict(bgcolor='white', font_size=12, font_family='Arial'),
    margin=dict(t=100, b=20)
)

fig.show()